# Colab setup

Runs this project on Google Colab. To open it: **File → Open notebook → GitHub**, search
`Luesh-coder/Lithium-Battery-ML-Project`, and pick `colab/colab_setup.ipynb`.

| Cell | When | What it does |
|---|---|---|
| 1 | every session | mounts Drive, clones (or pulls) the repo, installs the package, links storage to Drive, runs the tests |
| 2 | first session only | downloads the raw data (8.3 GB) and builds `cells.pkl`; skips itself once `cells.pkl` is on Drive |
| 3 | whenever | trains the baselines and writes scores and figures to Drive |
| 4 | whenever | trains the 1D CNN and compares it with the baselines |

- **No venv.** Colab's Python already has the GPU build of PyTorch, NumPy 2, SciPy, pandas, h5py,
  scikit-learn and matplotlib, so Cell 1 only runs `pip install -e .`.
- **Storage.** Colab's disk is wiped when the runtime disconnects. Cell 1 replaces
  `data/preprocessed/` and `results/` with links to `MyDrive/battery-ml/` on Google Drive.
  The raw `.mat` files stay on the Colab disk because they are too big for a free Drive.
- **Runtime.** A CPU runtime is enough for the baselines and the CNN, which trains on only
  41 cells. A GPU (Runtime → Change runtime type → T4 GPU) helps once you train many networks,
  such as the deep ensemble or a hyperparameter sweep.
- **Code changes.** Edit and push from your own machine; Cell 1 pulls the latest commit.
  Don't commit from this copy, because the Drive links replace tracked `.gitkeep` files.
- **Private repo.** Add a GitHub token as the Colab secret `GH_TOKEN` (key icon in the left sidebar)
  and allow this notebook to read it.

## 1. Setup (every session)

In [ ]:
import shutil
import sys
from pathlib import Path

from google.colab import drive, userdata

REPO_URL = "https://github.com/Luesh-coder/Lithium-Battery-ML-Project.git"
REPO = Path("/content/Lithium-Battery-ML-Project")
DRIVE_DIR = Path("/content/drive/MyDrive/battery-ml")  # survives disconnects

drive.mount("/content/drive")

# Clone on the first run of a session, pull on later runs.
try:
    token = userdata.get("GH_TOKEN")  # only needed if the repo is private
except Exception:
    token = None
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if (REPO / ".git").exists():
    !git -C {REPO} pull --ff-only
else:
    !git clone {url} {REPO}
%cd {REPO}

# No venv: install only this package into Colab's Python (pytest for the check below).
!pip install -q -e . pytest

# Keep preprocessed data and results on Drive. Only replaces real folders, never existing links.
for link, target in [
    (REPO / "data" / "preprocessed", DRIVE_DIR / "preprocessed"),
    (REPO / "results", DRIVE_DIR / "results"),
]:
    target.mkdir(parents=True, exist_ok=True)
    if not link.is_symlink():
        shutil.rmtree(link, ignore_errors=True)
        link.symlink_to(target, target_is_directory=True)
    print(f"{link.relative_to(REPO)} -> {target}")

# Scripts run with !python see the editable install; this lets notebook cells
# `import battery_ml` too, without restarting the session.
src = str(REPO / "src")
if src not in sys.path:
    sys.path.insert(0, src)

import numpy
import torch

assert int(numpy.__version__.split(".")[0]) >= 2, "battery_ml needs NumPy >= 2 (np.trapezoid)"
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)"
print(f"Python {sys.version.split()[0]} | NumPy {numpy.__version__} | PyTorch {torch.__version__} | GPU: {gpu}")

!python -m pytest -q

## 2. Download and preprocess (first session only)

In [ ]:
from battery_ml.config import load_config

cfg = load_config()
raw_dir = cfg["paths"]["raw_dir"]  # Colab disk
cells_pkl = cfg["paths"]["preprocessed_dir"] / cfg["data"]["output_file"]  # Drive

# data.matr.io file id and size in bytes of each batch file in configs/default.yaml
DOWNLOADS = {
    "2017-05-12_batchdata_updated_struct_errorcorrect.mat": ("5c86c0b5fa2ede00015ddf66", 3025320241),
    "2017-06-30_batchdata_updated_struct_errorcorrect.mat": ("5c86bf13fa2ede00015ddd82", 2007331155),
    "2018-04-12_batchdata_updated_struct_errorcorrect.mat": ("5c86bd64fa2ede00015ddbb2", 3236690412),
}

if cells_pkl.exists():
    print(f"{cells_pkl} already exists; skipping download and preprocessing.")
else:
    for name in cfg["data"]["batch_files"].values():
        file_id, size = DOWNLOADS[name]
        path = raw_dir / name
        if not (path.exists() and path.stat().st_size == size):
            !wget -q --show-progress -O {path} https://data.matr.io/1/api/v1/file/{file_id}/download
        assert path.stat().st_size == size, f"{name} is incomplete; re-run this cell"
    !python scripts/preprocess.py

## 3. Baselines

Runs the 100-cycle window for all three feature sets. The variance elastic net should give
RMSE ≈ 104 / 138 / 196 cycles (train / primary / secondary), matching Severson et al. Table 1.
Add `50 20 10` to `--windows` once their reference cycles are chosen in `configs/default.yaml`.
Scores go to `MyDrive/battery-ml/results/metrics/` and figures to `.../results/figures/`.

In [ ]:
for feature_set in ["variance", "discharge", "full"]:
    !python scripts/train_baselines.py --feature-set {feature_set} --windows 100
!python scripts/plot_baselines.py --window 100

## 4. 1D CNN

Trains the CNN on ΔQ(V) curves alone, then again with the Severson `full` features added to
its dense head, and prints both next to the baselines from Cell 3. The network is small, so a
CPU runtime is fine. Scores go to `results/metrics/cnn.csv` and `cnn_full.csv`, the loss per
epoch to `cnn*_history.csv`, and checkpoints to `results/models/`. Its settings are under
`cnn:` in `configs/default.yaml`. Edit them on your own machine, push, and re-run Cell 1.

In [ ]:
import pandas as pd

!python scripts/train_cnn.py --windows 100
!python scripts/train_cnn.py --windows 100 --scalar-features full

# RMSE (cycles) at the 100-cycle window: CNN vs baselines
metrics = REPO / "results" / "metrics"
runs = ["baselines_variance", "baselines_discharge", "baselines_full", "cnn", "cnn_full"]
scores = pd.concat(
    pd.read_csv(metrics / f"{run}.csv").assign(run=run) for run in runs if (metrics / f"{run}.csv").exists()
)
scores[scores.window == 100].pivot_table(index=["run", "model"], columns="split", values="rmse").round(1)